In [1]:
%matplotlib inline
import pandas as pd
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
#1. Download Data
class KaggleHouse(d2l.DataModule):
    def __init__(self, batch_size, train=None, val=None):
        super().__init__()
        self.save_hyperparameters()
        if self.train is None:
            self.raw_train = pd.read_csv(d2l.download(
                d2l.DATA_URL + 'kaggle_house_pred_train.csv', self.root,
                sha1_hash='585e9cc93e70b39160e7921475f9bcd7d31219ce'))
            self.raw_val = pd.read_csv(d2l.download(
                d2l.DATA_URL + 'kaggle_house_pred_test.csv', self.root,
                sha1_hash='fa19780a7b011d9b009e8bff8e99922a8ee2eb90'))

In [3]:
data = KaggleHouse(batch_size=64)
print(data.raw_train.shape)
print(data.raw_val.shape)

(1460, 81)
(1459, 80)


In [4]:
#2. Preprocess data
print(data.raw_train.iloc[:4, [0, 1, 2, 3, -3, -2, -1]])

   Id  MSSubClass MSZoning  LotFrontage SaleType SaleCondition  SalePrice
0   1          60       RL         65.0       WD        Normal     208500
1   2          20       RL         80.0       WD        Normal     181500
2   3          60       RL         68.0       WD        Normal     223500
3   4          70       RL         60.0       WD       Abnorml     140000


In [28]:
@d2l.add_to_class(KaggleHouse)
def preprocess(self):
    #Remove Id and Label columns from the data
    label = 'SalePrice'
    features = pd.concat(
        (
            self.raw_train.drop(columns=['Id', label]),
            self.raw_val.drop(columns=['Id'])
        )
    )

    #standardize numerical columns
    numeric_features = features.select_dtypes(include='number').columns.tolist()
    #print(features[numeric_features])
    features[numeric_features] = features[numeric_features].apply(lambda x: (x - x.mean() / x.std()))

    #replace NaN numberical features with 0
    features[numeric_features] = features[numeric_features].fillna(0)

    #Replace discrete features with one-hot encoding
    features = pd.get_dummies(features, dummy_na=True)

    #save preprocessed features
    self.train = features[:self.raw_train.shape[0]].copy()
    self.train[label] = self.raw_train[label]
    self.val = features[self.raw_train.shape[0]:].copy()

In [29]:
data.preprocess()
data.train.shape

(1460, 331)

In [52]:
@d2l.add_to_class(KaggleHouse)
def get_dataloader(self, train):
    label = 'SalePrice'
    data = self.train if train else self.val
    if label not in data: return
    get_tensor = lambda x: torch.tensor(x.values.astype(float), dtype=torch.float32)

    #logarithms of price as the loss function
    tensors = (get_tensor(data.drop(columns=[label])), #X
    torch.log(get_tensor(data[label])).reshape((-1, 1))) #Y
    #print(tensors)
    return self.get_tensorloader(tensors, train)

In [59]:
#K-fold cross validation
def k_fold_data(data, k):
    rets = []
    fold_size = data.train.shape[0] // k

    print('In function')
    for j in range(k):
        idx = range(j * fold_size, (j+1) * fold_size)
        rets.append(KaggleHouse(data.batch_size, data.train.drop(index=idx), data.train.loc[idx]))
    
    print(rets)
    return rets

In [67]:
def k_fold(trainer, data, k, lr):
    val_loss, models = [], []

    print('In k_fold')
    for i, data_fold in enumerate(k_fold_data(data, k)):
        model = d2l.LinearRegression(lr)
        model.board.yscale = 'log'
        if i!=0: model.board.display = False
        trainer.fit(model, data_fold)
        print(model.board.data['val_loss'])
        val_loss.append(float(model.board.data['val_loss'][-1].y))
        models.append(model)
    print(f'average validation log mse = {sum(val_loss)/len(val_loss)}')
    return models

In [68]:
trainer = d2l.Trainer(max_epochs=10)
models = k_fold(trainer, data, k=5, lr=0.01)

[Point(x=1.0, y=np.float32(nan)), Point(x=2.0, y=np.float32(nan)), Point(x=3.0, y=np.float32(nan)), Point(x=4.0, y=np.float32(nan)), Point(x=5.0, y=np.float32(nan)), Point(x=6.0, y=np.float32(nan)), Point(x=7.0, y=np.float32(nan)), Point(x=8.0, y=np.float32(nan)), Point(x=9.0, y=np.float32(nan)), Point(x=10.0, y=np.float32(nan))]
[Point(x=1.0, y=np.float32(nan)), Point(x=2.0, y=np.float32(nan)), Point(x=3.0, y=np.float32(nan)), Point(x=4.0, y=np.float32(nan)), Point(x=5.0, y=np.float32(nan)), Point(x=6.0, y=np.float32(nan)), Point(x=7.0, y=np.float32(nan)), Point(x=8.0, y=np.float32(nan)), Point(x=9.0, y=np.float32(nan)), Point(x=10.0, y=np.float32(nan))]
[Point(x=1.0, y=np.float32(nan)), Point(x=2.0, y=np.float32(nan)), Point(x=3.0, y=np.float32(nan)), Point(x=4.0, y=np.float32(nan)), Point(x=5.0, y=np.float32(nan)), Point(x=6.0, y=np.float32(nan)), Point(x=7.0, y=np.float32(nan)), Point(x=8.0, y=np.float32(nan)), Point(x=9.0, y=np.float32(nan)), Point(x=10.0, y=np.float32(nan))]
[Poi

ValueError: Data has no positive values, and therefore cannot be log-scaled.

ValueError: Data has no positive values, and therefore cannot be log-scaled.

<Figure size 350x250 with 1 Axes>